<a href="https://colab.research.google.com/github/chewanna7-code/NatureInsightStudy/blob/main/Compilment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
# ===== BLOCK 2: UPLOAD FILES IN GOOGLE COLAB =====
uploaded = files.upload()

Saving SS70-RP10.xlsx to SS70-RP10.xlsx
Saving SS70-RP10-ALL.xlsx to SS70-RP10-ALL.xlsx
Saving SS70-RP10-FI.xlsx to SS70-RP10-FI.xlsx
Saving SS70-RP10-LI.xlsx to SS70-RP10-LI.xlsx
Saving SS70-RP10-RI.xlsx to SS70-RP10-RI.xlsx
Saving SS70-RP20.xlsx to SS70-RP20.xlsx
Saving SS70-RP20-ALL.xlsx to SS70-RP20-ALL.xlsx
Saving SS70-RP20-FI.xlsx to SS70-RP20-FI.xlsx
Saving SS70-RP20-LI.xlsx to SS70-RP20-LI.xlsx
Saving SS70-RP20-RI.xlsx to SS70-RP20-RI.xlsx
Saving SS70-RP50.xlsx to SS70-RP50.xlsx
Saving SS70-RP50-ALL.xlsx to SS70-RP50-ALL.xlsx
Saving SS70-RP50-FI.xlsx to SS70-RP50-FI.xlsx
Saving SS70-RP50-LI.xlsx to SS70-RP50-LI.xlsx
Saving SS70-RP50-RI.xlsx to SS70-RP50-RI.xlsx
Saving SS70-RP100.xlsx to SS70-RP100.xlsx
Saving SS70-RP100-ALL.xlsx to SS70-RP100-ALL.xlsx
Saving SS70-RP100-FI.xlsx to SS70-RP100-FI.xlsx
Saving SS70-RP100-LI.xlsx to SS70-RP100-LI.xlsx
Saving SS70-RP100-RI.xlsx to SS70-RP100-RI.xlsx
Saving SS70-RP200.xlsx to SS70-RP200.xlsx
Saving SS70-RP200-ALL.xlsx to SS70-RP200-ALL

In [ ]:
# ===== BLOCK 3: DEFINE ORDERS AND LABELS =====
rp_order = ["RP10", "RP20", "RP50", "RP100", "RP200", "RP500"]

scenario_map = {
    "BASE": "No intervention",
    "LI": "Land use storage",
    "RI": "Runoff storage",
    "FI": "Floodplain storage",
    "ALL": "All interventions"
}

scenario_order = [
    "No intervention",
    "Land use storage",
    "Runoff storage",
    "Floodplain storage",
    "All interventions"
]

In [ ]:
# ===== BLOCK 4: PARSE FILE NAME =====
def parse_filename(filename):
    """
    Extract return period and scenario from filenames like:
    SS70-RP50.xlsx
    SS70-RP50-LI.xlsx
    SS70-RP50-FI.xlsx
    SS70-RP50-RI.xlsx
    SS70-RP50-ALL.xlsx
    """
    name = os.path.splitext(filename)[0]
    parts = name.split("-")

    rp = None
    scenario_code = "BASE"

    for part in parts:
        if re.fullmatch(r"RP\d+", part):
            rp = part

    if len(parts) >= 3:
        last_part = parts[-1]
        if last_part in ["LI", "RI", "FI", "ALL"]:
            scenario_code = last_part

    scenario = scenario_map.get(scenario_code, None)

    return rp, scenario

In [ ]:
# ===== BLOCK 5: OPTIONAL COLUMN CHECK =====
example_file = list(uploaded.keys())[0]
example_df = pd.read_excel(example_file)

print("Columns in example file:")
print(example_df.columns.tolist())

Columns in example file:
['Rain (Time in Sec.)', 'Rain (Intensity in mm/hour)', 'Unnamed: 2', 'Baseflow (Time in Sec.)', 'Baseflow (Flow in m3/s)', 'Unnamed: 5', 'Runoff (Time in Sec.)', 'Runoff (Flow in m3/s)', 'Unnamed: 8', 'Total (Time in Sec.)', 'Total (Flow in m3/s)']


In [ ]:
# ===== BLOCK 6: EXTRACT METRICS FROM ONE FILE =====
def extract_hydrograph_metrics(filepath):
    """
    Reads one Excel hydrograph file and extracts:
    - peak flow
    - time to peak (seconds)
    - time to peak (hours)
    """
    df = pd.read_excel(filepath)

    time_col = "Total (Time in Sec.)"
    flow_col = "Total (Flow in m3/s)"

    df = df[[time_col, flow_col]].dropna()

    peak_idx = df[flow_col].idxmax()

    peak_flow = df.loc[peak_idx, flow_col]
    time_to_peak_sec = df.loc[peak_idx, time_col]
    time_to_peak_hr = time_to_peak_sec / 3600

    return peak_flow, time_to_peak_sec, time_to_peak_hr

In [ ]:
# ===== BLOCK 7: READ ALL FILES AND STORE RESULTS =====
results = []

for file in uploaded.keys():
    if file.endswith(".xlsx") and not file.startswith("~$"):
        rp, scenario = parse_filename(file)

        if rp is None or scenario is None:
            print(f"Skipped: {file} (could not parse filename)")
            continue

        try:
            peak_flow, time_to_peak_sec, time_to_peak_hr = extract_hydrograph_metrics(file)

            results.append({
                "File": file,
                "Scenario": scenario,
                "Return Period": rp,
                "Peak Flow (m3/s)": peak_flow,
                "Time to Peak (sec)": time_to_peak_sec,
                "Time to Peak (hr)": time_to_peak_hr
            })

        except Exception as e:
            print(f"Error reading {file}: {e}")

In [ ]:
# ===== BLOCK 8: CREATE MASTER RESULTS DATAFRAME =====
results_df = pd.DataFrame(results)

results_df["Scenario"] = pd.Categorical(results_df["Scenario"], categories=scenario_order, ordered=True)
results_df["Return Period"] = pd.Categorical(results_df["Return Period"], categories=rp_order, ordered=True)

results_df = results_df.sort_values(["Scenario", "Return Period"]).reset_index(drop=True)

results_df

,File,Scenario,Return Period,Peak Flow (m3/s),Time to Peak (sec),Time to Peak (hr)
0,SS70-RP10.xlsx,No intervention,RP10,109.606944,83400,23.166667
1,SS70-RP20.xlsx,No intervention,RP20,125.154148,83400,23.166667
2,SS70-RP50.xlsx,No intervention,RP50,147.269127,83400,23.166667
3,SS70-RP100.xlsx,No intervention,RP100,165.450901,83400,23.166667
4,SS70-RP200.xlsx,No intervention,RP200,185.064302,83400,23.166667
5,SS70-RP500.xlsx,No intervention,RP500,213.438585,83400,23.166667
6,SS70-RP10-LI.xlsx,Land use storage,RP10,109.606944,83400,23.166667
7,SS70-RP20-LI.xlsx,Land use storage,RP20,125.154148,83400,23.166667
8,SS70-RP50-LI.xlsx,Land use storage,RP50,147.269127,83400,23.166667
9,SS70-RP100-LI.xlsx,Land use storage,RP100,165.450901,83400,23.166667


In [ ]:
# ===== BLOCK 9: RAW PEAK FLOW TABLE =====
peak_flow_table = results_df.pivot(
    index="Scenario",
    columns="Return Period",
    values="Peak Flow (m3/s)"
)

peak_flow_table = peak_flow_table.reindex(index=scenario_order, columns=rp_order)

peak_flow_table

Return Period,RP10,RP20,RP50,RP100,RP200,RP500
Scenario,,,,,,
No intervention,109.606944,125.154148,147.269127,165.450901,185.064302,213.438585
Land use storage,109.606944,125.154148,147.269127,165.450901,185.064302,213.438585
Runoff storage,101.316763,114.443509,138.894753,165.450901,185.064302,213.438585
Floodplain storage,107.622048,122.479014,143.517263,160.736248,179.240562,205.895788
All interventions,99.117745,111.775194,133.203415,157.388955,177.217021,204.821916


In [ ]:
# ===== BLOCK 10: RAW TIME TO PEAK TABLE =====
time_to_peak_table = results_df.pivot(
    index="Scenario",
    columns="Return Period",
    values="Time to Peak (hr)"
)

time_to_peak_table = time_to_peak_table.reindex(index=scenario_order, columns=rp_order)

time_to_peak_table

Return Period,RP10,RP20,RP50,RP100,RP200,RP500
Scenario,,,,,,
No intervention,23.166667,23.166667,23.166667,23.166667,23.166667,23.166667
Land use storage,23.166667,23.166667,23.166667,23.166667,23.166667,23.166667
Runoff storage,23.833333,23.833333,25.000000,23.166667,23.166667,23.166667
Floodplain storage,23.500000,23.500000,23.500000,23.500000,23.500000,23.500000
All interventions,24.166667,24.166667,25.166667,23.833333,23.666667,23.666667


In [ ]:
# ===== BLOCK 11: DIFFERENCE TABLES RELATIVE TO BASELINE =====
baseline_peak = peak_flow_table.loc["No intervention"]
baseline_time = time_to_peak_table.loc["No intervention"]

peak_diff_table = peak_flow_table.subtract(baseline_peak, axis=1)
time_diff_table = time_to_peak_table.subtract(baseline_time, axis=1)

peak_diff_table

Return Period,RP10,RP20,RP50,RP100,RP200,RP500
Scenario,,,,,,
No intervention,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00
Land use storage,-2.842171e-14,0.000000,0.000000,5.684342e-14,0.000000,5.684342e-14
Runoff storage,-8.290181e+00,-10.710638,-8.374373,0.000000e+00,0.000000,5.684342e-14
Floodplain storage,-1.984896e+00,-2.675134,-3.751864,-4.714653e+00,-5.823741,-7.542796e+00
All interventions,-1.048920e+01,-13.378954,-14.065712,-8.061946e+00,-7.847282,-8.616668e+00


In [ ]:
time_diff_table

Return Period,RP10,RP20,RP50,RP100,RP200,RP500
Scenario,,,,,,
No intervention,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Land use storage,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
Runoff storage,0.666667,0.666667,1.833333,0.000000,0.000000,0.000000
Floodplain storage,0.333333,0.333333,0.333333,0.333333,0.333333,0.333333
All interventions,1.000000,1.000000,2.000000,0.666667,0.500000,0.500000


In [ ]:
# ===== BLOCK 12: OPTIONAL PERCENT CHANGE IN PEAK FLOW =====
peak_percent_change_table = ((peak_flow_table - baseline_peak) / baseline_peak) * 100

peak_percent_change_table

Return Period,RP10,RP20,RP50,RP100,RP200,RP500
Scenario,,,,,,
No intervention,0.000000e+00,0.000000,0.000000,0.000000e+00,0.000000,0.000000e+00
Land use storage,-2.593057e-14,0.000000,0.000000,3.435667e-14,0.000000,2.663221e-14
Runoff storage,-7.563554e+00,-8.557957,-5.686442,0.000000e+00,0.000000,2.663221e-14
Floodplain storage,-1.810922e+00,-2.137471,-2.547624,-2.849578e+00,-3.146874,-3.533942e+00
All interventions,-9.569830e+00,-10.689980,-9.551025,-4.872712e+00,-4.240300,-4.037071e+00


In [ ]:
# ===== BLOCK 13: EXPORT TO EXCEL =====
output_file = "Hydrograph_Summary_Tables.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Raw Extracted Data", index=False)
    peak_flow_table.to_excel(writer, sheet_name="Peak Flow Raw")
    time_to_peak_table.to_excel(writer, sheet_name="Time to Peak Raw")
    peak_diff_table.to_excel(writer, sheet_name="Peak Flow Difference")
    time_diff_table.to_excel(writer, sheet_name="Time Difference")
    peak_percent_change_table.to_excel(writer, sheet_name="Peak Flow % Change")

files.download(output_file)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>